In [1]:
import copy
import numpy as np

import mrmustard.lab as mr
import mrmustard.physics as mrph
import mrmustard.math as mrmath
from mrmustard import settings as mrsettings

from generalized_photon_catalysis.quadratic_preparation import generalized_e2_preparation
from generalized_photon_catalysis.preparation_circuit import get_product_form_preparation_circuit, mra, mra_dag
from photon_catalysis.utils import state_array_to_dict, state_to_string

mrsettings.AUTOSHAPE_MAX = 4
mrsettings.DEFAULT_FOCK_SIZE = 4
mrmath.change_backend("jax")

In [2]:
r = 0.08
sh = np.sinh(2*r)
ch = np.cosh(2*r)
M = 2
A = np.array([
    [0, sh, 0, ch],
    [sh, 0, ch, 0],
    [0, ch, 0, sh],
    [ch, 0, sh, 0]
])

inp = np.zeros((3, 3))
inp[1, 1] = 1
inp = mr.Ket.from_fock((1, 2), inp)

target = np.zeros((3, 3))
target[2, 0] = np.sqrt(2)*ch
target[0, 2] = np.sqrt(2)*ch
target[2, 2] = 2*sh
target[0, 0] = sh
target = mr.Ket.from_fock((1, 2), array=target).normalize()

In [3]:
M = 2
A = np.array([
    [0, 0, 0, 1],
    [0, 0, -1, 0],
    [0, -1, 0, 0],
    [1, 0, 0, 0]
])

inp = np.zeros((3, 3))
inp[1, 2] = 1
inp = mr.Ket.from_fock((1, 2), inp)

target = np.zeros((4, 4))
target[2, 1] = 2
target[0, 3] = -np.sqrt(3)
target = mr.Ket.from_fock((1, 2), array=target).normalize()

In [4]:
V = generalized_e2_preparation(A, 5000, 1e-3)

step    0  loss = 1.078121  |O^T O - I| = 5.23e-07
step  200  loss = 0.008903  |O^T O - I| = 2.84e-07
step  400  loss = 0.006954  |O^T O - I| = 3.51e-07
step  600  loss = 0.005424  |O^T O - I| = 2.16e-07
step  800  loss = 0.004223  |O^T O - I| = 2.13e-07
step 1000  loss = 0.003278  |O^T O - I| = 2.97e-07
step 1200  loss = 0.002529  |O^T O - I| = 1.48e-07
step 1400  loss = 0.001930  |O^T O - I| = 3.53e-07
step 1600  loss = 0.001450  |O^T O - I| = 3.48e-07
step 1800  loss = 0.001061  |O^T O - I| = 4.00e-07
step 2000  loss = 0.000746  |O^T O - I| = 2.74e-07
step 2200  loss = 0.000490  |O^T O - I| = 1.63e-07
step 2400  loss = 0.000282  |O^T O - I| = 3.26e-07
step 2600  loss = 0.000112  |O^T O - I| = 4.55e-07
step 2800  loss = 0.000001  |O^T O - I| = 3.24e-07
step 3000  loss = 0.000000  |O^T O - I| = 2.49e-07
step 3200  loss = 0.000000  |O^T O - I| = 4.89e-07
step 3400  loss = 0.000000  |O^T O - I| = 2.06e-07
step 3600  loss = 0.000000  |O^T O - I| = 4.59e-07
step 3800  loss = 0.000000  |O^

In [5]:
V_anc = np.zeros((V.shape[0], 2*M + 2), dtype=np.complex128)
for i in range(V.shape[0]):
    v = V[i]
    V_anc[i] = np.concat([[1], v[:M], [0], v[M:]], dtype=np.complex64)

G0, ancillas = get_product_form_preparation_circuit(V_anc, r=0.01, theta=0.01)
G = copy.deepcopy(G0)
print(ancillas)
for a in ancillas:
    G = mr.Vacuum((a)) >> G >> mr.Number(a, 1).dual

Dist to symplectic: 1.5028569429463127e-14, op: -1.0
Dist to symplectic: 2.1748260365286893e-12, op: -1.0
Dist to symplectic: 1.057945177454471e-11, op: -1.0
Dist to symplectic: 2.3109921054406596e-10, op: 1.0
[3, 4, 5, 6]


/home/andrew/Documents/Sorbonne/M2_2/catalysis-experimental/.venv/lib/python3.10/site-packages/mrmustard/physics/ansatz/array_ansatz.py:260: UserWarning: The fock array is being padded with zeros. Is this really necessary?
  warn(


In [10]:
g_inp = mr.SqueezedVacuum(1, 0.1) >> mr.SqueezedVacuum(2, 0.01)

In [11]:
target_op = (mra_dag(1) >> mra(2)) - (mra_dag(2) >> mra(1))
target = g_inp >> target_op
target = target.normalize()

In [12]:
r = mr.Vacuum(0) >> g_inp >> G >> mr.Number(0, 2).dual
r = r.normalize()

In [13]:
fid = np.float64(target.fidelity(r).real)
fid

0.982018787560163